In [7]:
import pandas as pd
import numpy as np
import os

# get data with for loop


In [8]:

folder_path = r'D:\olist-ecommerce-analytics\data\raw'

df={}

for file_name in os.listdir(folder_path):
    if file_name.endswith('.csv'):
        file_path = os.path.join(folder_path, file_name)
        clean = file_name.replace('.csv', '')
        
        df[clean] = pd.read_csv(file_path)
    else:
        print(f"Skipping non-CSV file: {file_name}")    

Skipping non-CSV file: .gitkeep


In [9]:
df.keys()

dict_keys(['customers', 'olist_geolocation_dataset', 'olist_orders_dataset', 'olist_order_items_dataset', 'olist_order_payments_dataset', 'olist_order_reviews_dataset', 'olist_products_dataset', 'olist_sellers_dataset', 'product_category_name_translation'])

# Why is the `globals()` loop commented out?

### The Idea:
- When you have many tables (like `df['customers']`, `df['orders']`), it might seem helpful to extract them into standalone variables (`customers`, `orders`) so you don't have to type `df['...']` every time.

### Why is this a bad practice? (Why we don't uncomment it):
- **Hard to Debug:** It creates variables dynamically out of thin air. If you make a typo, it's very hard to track down where the variable came from or what it is.
- **Memory Heavy:** It duplicates references to large DataFrames in the global namespace, cluttering the memory.
- **Best Practice:** Keep them inside the `df` dictionary. Access them as `df['customers']`. It is cleaner and keeps all your data in one place.

In [10]:
##for name , data in df.items():
    ##globals()[name] = data

# Create Function: `sum_nulls`

### Why `df.isnull().sum()`?
- `df.isnull()` returns a DataFrame of the exact same shape, but with `True` for empty cells and `False` for filled cells.
- Adding `.sum()` along the columns (axis 0 by default) adds up all the `True` values for each column. This instantly gives us the count of missing values per column without needing a loop.

### Why `len(df)` instead of `df.count()`?
- **Crucial point:** `df.count()` ignores `NaN` (missing) values and returns a smaller number if the dataset is incomplete.
- `len(df)` always returns the exact total number of rows in the DataFrame.
- To calculate a correct **percentage** of missing values, you must divide by the **total** number of rows (`len(df)`).

### Why `display()` instead of `print()`?
- In Jupyter Notebook, `display()` renders a clean, interactive HTML table.
- `print()` converts the DataFrame to a plain text string, which can cut off columns or look messy.

### Why are the values wrapped in square brackets `[count]`?
- When creating a DataFrame from a dictionary, Pandas usually expects values to be **lists** (columns).
- By wrapping the scalar values in `[ ]` (e.g., `[count]`), we force Pandas to treat them as a **single row** rather than trying to expand them into a long column.

### Why build it using a dictionary `{'count': ...}`?
- This standardizes the output format perfectly with your other helper functions (`sum_dup` and `sum_whitespace`), making the final `data_quality_report` clean and consistent.

In [11]:
def sum_nulls(df):
     """
     Calculates the total count and percentage of null values for each column in a DataFrame.

     Parameters:
     df (pd.DataFrame): The input pandas DataFrame to analyze.

     Returns:
     pd.DataFrame: A DataFrame containing 'count' and 'null_percent' for each column.
     """
    
     count = df.isnull().sum()
     null_percent = (count / len(df)) * 100
     display(pd.DataFrame({'count': count, 'null_percent': null_percent}))

# Test Before Create

### Why `.duplicated().sum()` instead of `.duplicated().count()`?
- `duplicated()` returns a boolean **Series** (`True` for duplicates, `False` for unique).
- In Python, `True` is treated as `1` and `False` as `0`.
- Therefore, `.sum()` adds up all the `True` values, giving us exactly the number of duplicates.
- Using `.count()` would count *all* rows (including the non-duplicates), which is not what we want.

### Why does it return `np.int64(0)`?
- Pandas is built on NumPy, so it returns NumPy integer types. (It's perfectly fine, but you can wrap it in `int()` if you prefer a standard Python integer).

In [12]:
df['customers'].duplicated().sum()

np.int64(0)

# Create Master Function: `sum_dup`

### Why `.duplicated().sum()`?
- Counts the total number of duplicated rows efficiently.

### Why `len(df)` instead of `df.count()` for the percentage?
- **Crucial point:** `df.count()` ignores `NaN` (missing) values and returns a smaller number if the dataset is incomplete.
- `len(df)` always returns the exact total number of rows in the DataFrame. 
- To calculate a correct **percentage** of duplicates, you must divide by the **total** number of rows (`len(df)`).

### Why `display()` instead of `print()`?
- In Jupyter Notebook, `display()` renders a clean, interactive HTML table.
- `print()` converts the DataFrame to a plain text string, which can cut off columns or look messy.

### Why are the values wrapped in square brackets `[count]`?
- When creating a DataFrame from a dictionary, Pandas usually expects values to be **lists** (columns).
- By wrapping the scalar values in `[ ]` (e.g., `[count]`), we force Pandas to treat them as a **single row** rather than trying to expand them into a long column.

### Why build it using a dictionary `{'count': ...}`?
- This standardizes the output format perfectly with your other helper functions (`sum_whitespace` and `sum_nulls`), making the final `data_quality_report` clean and consistent.

In [13]:
def sum_dup(df):
    """
    Calculates the total count and percentage of duplicated rows in a DataFrame.

    Parameters:
    df (pd.DataFrame): The input pandas DataFrame to analyze.

    Returns:
    pd.DataFrame: A DataFrame containing 'count' and 'dup_percent' for duplicated rows.
    """
    
    count = df.duplicated().sum()
    dup_percent = round((count / len(df)) * 100, 2)
    display(pd.DataFrame({'count': [count], 'dup_percent': [dup_percent]}))

### test func

In [14]:
sum_dup(df['customers'])

,count,dup_percent
0,0,0.0


### here this is return whitespace something like trim in sql 

In [15]:
(df['customers']['customer_id'].str.strip() != df['customers']['customer_id']).sum()

np.int64(0)

# Create Function: `sum_whitespace`

### Why `df.select_dtypes(include=['object', 'string']).columns`?
- **Crucial point:** The `.str` accessor only works on text columns. If you try to run `.str.strip()` on a numeric column, Python will throw an `AttributeError`.
- `select_dtypes` filters the DataFrame *before* the loop, ensuring we only apply string operations to columns that actually contain strings (objects).

### Why `.fillna('')`?
- **Crucial point:** Pandas treats missing values (`NaN`) as `NaN`. If a cell is `NaN`, comparing `NaN != NaN` evaluates to `True`, which would incorrectly flag it as having "whitespace".
- By replacing `NaN` with an empty string `''`, the comparison `'' != ''` becomes `False`. This perfectly isolates only the cells that actually contain extra spaces.

### Why `.str.strip() != df[col_name]`?
- `strip()` removes leading and trailing spaces (e.g., `" Python "` becomes `"Python"`).
- We compare the stripped version against the original. If they are not equal `!=`, it means the original had extra spaces that were removed.
- **Alternative:** `df[col_name].str.strip()` is preferred over `df[col_name].str.replace(' ', '')` because `replace` removes *all* spaces (including spaces between words), while `strip` only removes the edges.

### Why `round(..., 2)`?
- It limits the percentage to 2 decimal places, making the final report much cleaner and easier to read.

### Why `int(white_space)` inside the dictionary?
- As we learned earlier, Pandas may coerce the `count` column to a Float if the `percent` column is a Float.
- Wrapping the count in `int()` ensures the count is displayed as a clean whole number (e.g., `5`) instead of `5.0`.

### Why `.T` at the end?
- When creating `pd.DataFrame(whitespace_counts)`, the column names become the keys, and the statistics (`count` and `percent`) become the index (rows).
- Using `.T` transposes the table. This is the standard, readable format for quality reports: **Column Name in the Index, and Count/Percent as Columns**.

In [16]:
def sum_whitespace(df):
    
    
    whitespace_counts = {}
      
      
    """
    Calculates the total count and percentage of values with leading or trailing whitespace for each column in a DataFrame.

    Parameters:
    df (pd.DataFrame): The input pandas DataFrame to analyze.

    Returns:
    pd.DataFrame: A DataFrame containing 'count' and 'whitespace_percent' for each column.
    """
    
    
    for col_name in df.select_dtypes(include=['object', 'string']).columns:
        white_space = (df[col_name].fillna('').str.strip() != df[col_name].fillna('')).sum()
        white_space_percent = round((white_space / len(df)) * 100, 2)
        whitespace_counts[col_name] = {'count':int(white_space), 'whitespace_percent': white_space_percent}
    
    display(pd.DataFrame(whitespace_counts).T)
    





In [17]:
sum_whitespace(df['customers'])

,count,whitespace_percent
customer_id,0.0,0.0
customer_unique_id,0.0,0.0
customer_city,0.0,0.0
customer_state,0.0,0.0


# Create Master Function: `data_quality_report`

### Why create a Master Function?
- **Modularity (DRY Principle):** Instead of manually calling `sum_dup`, `sum_whitespace`, and `sum_nulls` every single time for every table, we write a "wrapper" function that calls them all for us. This makes the notebook much cleaner and faster to run.
- **Consistency:** It ensures every dataframe gets the exact same quality checks applied in the exact same order every single time.

### Why `print(...)` for the headers and separators?
- Headers like `---shape---` and separators like `"="*50` are **strings**, not DataFrames. `display()` is specifically meant for rendering data tables (like our `sum_dup` function does). So we use `print` for text to keep it aligned.
- **The Separators (`"="*50`):** We use `print("\n" + "="*50 + "\n")` to create visual breaks between sections. The `"\n"` adds empty lines (spacing) to make the output less cluttered. The `"="*50` draws a line across the screen so you can easily see where one check ends and the next begins.

### Why `print(df.shape)`?
- `df.shape` returns a tuple (e.g., `(100, 5)`). Since it is the first line in the function, simply writing `df.shape` wouldn't show anything in the output. We MUST wrap it in `print()` if we want to see it!

### Why `df.info()` WITHOUT `print()`?
- **Crucial point:** The `df.info()` method in Pandas prints its output *directly to the console* and returns `None`. 
- If you wrote `print(df.info())`, Pandas would print the table, and then the `print` statement would print the word `None` right below it. This adds ugly, unnecessary clutter to your output. So, we keep it as `df.info()`.

### Why call `sum_dup(df)`, `sum_whitespace(df)`, and `sum_nulls(df)`?
- These functions are designed to handle the heavy lifting (computing counts, percentages, and displaying DataFrames). 
- **Note:** These are currently using `display()` internally to render the output as pretty tables. It's perfect to just call them without a `print` wrapper, because the `display()` call inside them handles the output.

In [18]:
def data_quality_report(df):
    """Prints a comprehensive summary of data quality for a DataFrame (Info, Duplicates, Whitespace, Nulls)."""
    
    
    print("--- Data Quality Report ---")
    print("\n" + "="*50 + "\n")
    print('---shape----')
    print(df.shape)
    print("\n" + "="*50 + "\n")
    print('---info---')
    df.info()
    print("\n" + "="*50 + "\n")
    print("--- Duplicate Check ---")
    sum_dup(df)
    print("\n" + "="*50 + "\n")
    print("\n--- Whitespace Check ---")
    sum_whitespace(df)
    print("\n--- Missing Values Check ---")
    sum_nulls(df)
    

In [19]:
data_quality_report(df['customers'])

--- Data Quality Report ---


---shape----
(99441, 5)


---info---
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 11.0 MB


--- Duplicate Check ---


,count,dup_percent
0,0,0.0





--- Whitespace Check ---


,count,whitespace_percent
customer_id,0.0,0.0
customer_unique_id,0.0,0.0
customer_city,0.0,0.0
customer_state,0.0,0.0



--- Missing Values Check ---


,count,null_percent
customer_id,0,0.0
customer_unique_id,0,0.0
customer_zip_code_prefix,0,0.0
customer_city,0,0.0
customer_state,0,0.0


# 💡 Why I am inspecting manually instead of using a For Loop

Although I have built a powerful `data_quality_report` Master Function, I am choosing to call it **table by table (manual inspection)** instead of running a `for` loop across all tables at once.

### 1. Understanding the "Why" behind the dirt (Context is King)
A `for` loop prints numbers, but it does not explain them. 
For example, the `orders` table has `NaN` values in delivery dates. If I saw this in a giant combined loop, I might think it was an error and try to drop or fill them. Inspecting it manually allows me to realize: **"These nulls are valid business logic; they simply mean the order hasn't been delivered yet!"**

### 2. Avoiding Blind Automation (Risk of Data Loss)
Different tables have different rules:
- **Customers:** Duplicates = Errors (Safe to drop).
- **Order Items:** Duplicates = Multiple products in one order (NOT an error. Dropping them would break the data!).
- Manual inspection prevents me from writing a generic "cleanup script" that accidentally destroys valid business data.

### 3. Focus and Cognitive Load
A `for` loop dumps 4 massive reports onto the screen at once. It is overwhelming and easy to skip over important details. Inspecting one table at a time keeps my focus sharp and allows me to fully absorb the characteristics of that specific dataset.

### 4. Building Analytical Intuition
Since this is the initial EDA phase, I want to see the actual raw data (`df.head()`), understand the column meanings, and build a mental model of the data. This deep analytical intuition is built through manual exploration, not bulk automation.

---

*Note: The Master Function is still extremely valuable! I will use it as a dedicated "spot-check" tool for individual tables as I move through the cleaning process.*

In [20]:
data_quality_report(df['olist_geolocation_dataset'])

--- Data Quality Report ---


---shape----
(1000163, 5)


---info---
<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(2)
memory usage: 50.1 MB


--- Duplicate Check ---


,count,dup_percent
0,261831,26.18





--- Whitespace Check ---


,count,whitespace_percent
geolocation_city,1.0,0.0
geolocation_state,0.0,0.0



--- Missing Values Check ---


,count,null_percent
geolocation_zip_code_prefix,0,0.0
geolocation_lat,0,0.0
geolocation_lng,0,0.0
geolocation_city,0,0.0
geolocation_state,0,0.0


# 📊 Data Quality Report: `df['olist_geolocation_dataset']`

### 💡 Key Insights & Why Manual Inspection Matters Here:

**1. The "Valid" Duplicates (26%):**
If I had run a `for` loop and seen 26% duplicates, I might have blindly deleted them. **DO NOT DO THAT!**
In the Olist dataset, `geolocation_zip_code_prefix` is NOT a unique key. Multiple rows share the same zip code prefix because they represent different lat/lng coordinates for that region (often covering different streets or neighborhoods). These are **Valid Duplicates** based on business logic.
*   **Action:** You cannot simply drop duplicates. You must **Aggregate** them. (e.g., `df.groupby('geolocation_zip_code_prefix').agg({'geolocation_lat': 'mean', 'geolocation_lng': 'mean'})` to get the center of that zip code for joining).

**2. The 1 Whitespace Error:**
The `whitespace_percent` shows `1.0` in `geolocation_c` (city). This is an easy fix: `df['geolocation_city'] = df['geolocation_city'].str.strip()`. 

**3. Data Types (Float64 vs Int64):**
Notice `geolocation_lat` and `geolocation_lng` are `float64`. This is correct because coordinates require decimals. 
*   **Caution:** Be careful with `geolocation_zip_code_prefix` as `int64`! If a zip code starts with a `0` (like `01234`), Pandas will drop the leading zero, which can break your join with the `customers` table. It is often safer to convert zip codes to `str` (strings) or pad them later.

---

### 🚀 Next Step:
Clean this table by:
1.  Fixing the whitespace in the city column.
2.  Aggregating the lat/lng coordinates by `geolocation_zip_code_prefix` (so you have exactly 1 row per zip code).
3.  Converting the zip code to string to preserve leading zeros.

In [21]:
data_quality_report(df[ 'olist_orders_dataset'])

--- Data Quality Report ---


---shape----
(99441, 8)


---info---
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 21.9 MB


--- Duplicate Check ---


,count,dup_percent
0,0,0.0





--- Whitespace Check ---


,count,whitespace_percent
order_id,0.0,0.0
customer_id,0.0,0.0
order_status,0.0,0.0
order_purchase_timestamp,0.0,0.0
order_approved_at,0.0,0.0
order_delivered_carrier_date,0.0,0.0
order_delivered_customer_date,0.0,0.0
order_estimated_delivery_date,0.0,0.0



--- Missing Values Check ---


,count,null_percent
order_id,0,0.000000
customer_id,0,0.000000
order_status,0,0.000000
order_purchase_timestamp,0,0.000000
order_approved_at,160,0.160899
order_delivered_carrier_date,1783,1.793023
order_delivered_customer_date,2965,2.981668
order_estimated_delivery_date,0,0.000000


# 📊 Data Quality Report: `df['olist_orders_dataset']`
---

### 💡 Key Insights & "The Why" behind the Missing Values:

**1. The "Valid" Lifecycle Nulls:**
If I had run a `for` loop and seen the 2,965 nulls in `delivered_customer_date`, I might have dropped them. **DO NOT DO THAT!**
These nulls represent **orders that have not reached that stage yet** (e.g., still in transit, pending approval, or canceled). It is a normal progression of the business.
*   `order_approved_at` (160 missing): The payment hasn't been approved yet.
*   `order_delivered_carrier_date` (1783 missing): Order is still waiting to be handed to the carrier.
*   `order_delivered_customer_date` (2965 missing): The customer hasn't received the package yet.

**2. The Critical Bug: Data Types (Strings vs Dates):**
Look at the `Info` output: `order_purchase_timestamp`, `order_estimated_delivery_date`, etc., are all **`str`**!
If I leave them as strings, I cannot calculate "Delivery Time" (e.g., `delivered_date - purchase_date`) or find "Late Orders" (e.g., `delivered_date > estimated_date`).
*   **Action:** I must convert all these columns to `pd.to_datetime()`.

---

### 🚀 Next Step (The Cleaning Phase for this table):
1. **Do nothing with the nulls** (they are valid business logic).
2. **Convert all timestamp columns to `datetime`:** `df['orders'][col] = pd.to_datetime(df['orders'][col])`
3. **Check `order_status`:** The status column (e.g., `delivered`, `canceled`, `shipped`) will explain *exactly* why the nulls exist in the late-stage columns!

In [22]:
data_quality_report(df['olist_order_items_dataset'])

--- Data Quality Report ---


---shape----
(112650, 7)


---info---
<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 18.4 MB


--- Duplicate Check ---


,count,dup_percent
0,0,0.0





--- Whitespace Check ---


,count,whitespace_percent
order_id,0.0,0.0
product_id,0.0,0.0
seller_id,0.0,0.0
shipping_limit_date,0.0,0.0



--- Missing Values Check ---


,count,null_percent
order_id,0,0.0
order_item_id,0,0.0
product_id,0,0.0
seller_id,0,0.0
shipping_limit_date,0,0.0
price,0,0.0
freight_value,0,0.0


# 📊 Data Quality Report: `df['olist_order_items_dataset']`

### 💡 Key Insights & "The Why" behind this table:

**1. The "False" Duplicates (Why `order_id` MUST repeat):**
If you see `order_id` appear multiple times, **DO NOT DROP IT!** This is a "Fact" table. An order can contain multiple items (e.g., a customer buys 3 different products in one order).
*   The Unique Key is actually **(`order_id` + `order_item_id`)**. `order_item_id` is `1` for the first item, `2` for the second, etc. Therefore, the exact row is never duplicated.

**2. The Date Conversion:**
Just like the `orders` table, `shipping_limit_date` is currently stored as a `str`.
*   **Action:** Convert it to `pd.to_datetime()` to calculate shipping lateness or delivery times in future steps.

**3. The Numeric Columns:**
`price` and `freight_value` are correctly stored as `float64`. This is perfect for mathematical aggregations (e.g., calculating Average Order Value - AOV).

In [23]:
data_quality_report(df['olist_order_payments_dataset'])

--- Data Quality Report ---


---shape----
(103886, 5)


---info---
<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 8.1 MB


--- Duplicate Check ---


,count,dup_percent
0,0,0.0





--- Whitespace Check ---


,count,whitespace_percent
order_id,0.0,0.0
payment_type,0.0,0.0



--- Missing Values Check ---


,count,null_percent
order_id,0,0.0
payment_sequential,0,0.0
payment_type,0,0.0
payment_installments,0,0.0
payment_value,0,0.0


# 📊 Data Quality Report: `df['olist_order_payments_dataset']`

### 💡 Key Insights & "The Why" behind this table:

**1. The "False" Duplicates (Why `order_id` MUST repeat):**
Just like `order_items`, **DO NOT DROP DUPLICATES ON `order_id`!** 
This is a Fact table for transactions. An order can be split into multiple payment methods (e.g., customer pays half by credit card and half by voucher, or splits it into multiple sequential payments). 
*   `payment_sequential` tells us the order of those payments (1, 2, 3...). This is perfectly valid data.

**2. The `payment_type` and `installments`:**
*   `payment_type` is a categorical string (e.g., `credit_card`, `boleto`, `voucher`, `debit_card`). 
*   **Note:** There will be values like `not_defined` here. This is a valid category, not a mistake! 
*   `payment_installments` is an integer. **Do not convert it to float**, it represents the number of months (e.g., 1, 2, 3, 10).

**3. The Currency Column:**
`payment_value` is perfectly stored as `float64`. This is ideal for calculating Total Revenue, Average Order Value (AOV), or running aggregation functions.

---

### 🚀 Next Step:
This table requires **zero cleaning**. It is completely ready to be merged with your `orders` table using a Left Join on `order_id`. 

**The Master Cleanup (for all tables so far):**
You have successfully inspected the `orders` table (dates need conversion), `order_items`, and `payments`. The next step is to convert the `shipping_limit_date` in items and the timestamps in orders to `datetime`, and then start merging!

In [24]:
data_quality_report(df['olist_order_reviews_dataset'])

--- Data Quality Report ---


---shape----
(99224, 7)


---info---
<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 17.8 MB


--- Duplicate Check ---


,count,dup_percent
0,0,0.0





--- Whitespace Check ---


,count,whitespace_percent
review_id,0.0,0.00
order_id,0.0,0.00
review_comment_title,1998.0,2.01
review_comment_message,9451.0,9.52
review_creation_date,0.0,0.00
review_answer_timestamp,0.0,0.00



--- Missing Values Check ---


,count,null_percent
review_id,0,0.000000
order_id,0,0.000000
review_score,0,0.000000
review_comment_title,87656,88.341530
review_comment_message,58247,58.702532
review_creation_date,0,0.000000
review_answer_timestamp,0,0.000000


# 📊 Data Quality Report: `df['olist_order_reviews_dataset']`

### 💡 Key Insights & "The Why" behind the Missing Text:

**1. The "Valid" Nulls (The most important insight!):**
If I had run a `for` loop and seen 88% missing data in the comment title, I might have dropped the entire column or rows. **DO NOT DO THAT!**
*   **`review_score` (0% missing):** Customers are *mandated* to give a 1-5 rating.
*   **`review_comment_title` (88% missing) & `review_comment_message` (58% missing):** Customers are *not mandated* to write a text comment. They just hit the stars and click submit.
*   *Action:* Keep them as `NaN`. They are valid business logic representing "customers who didn't leave a text review".

**2. The Whitespace in Comments:**
The ~2,000 and ~9,400 counts in the whitespace report are rows that might just contain spaces, or spaces around the text.
*   *Action:* Run `df['reviews']['review_comment_message'] = df['reviews']['review_comment_message'].str.strip()` to clean the text.

**3. Data Types (Dates):**
Just like the `orders` table, `review_creation_date` and `review_answer_timestamp` are `str`.
*   *Action:* Convert them to `pd.to_datetime()` so you can calculate how long it took the seller/company to respond to the review.

**4. The Duplicates:**
`0.0%` duplicates. It is perfectly unique. No action needed.

---

### 🚀 Next Step:
Fix the whitespace, convert the date columns, and then merge this table to the `orders` table using `order_id` so you can analyze how shipping delays impact customer reviews!

In [25]:
data_quality_report(df[ 'olist_products_dataset'])

--- Data Quality Report ---


---shape----
(32951, 9)


---info---
<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 3.7 MB


--- Duplicate Check ---


,count,dup_percent
0,0,0.0





--- Whitespace Check ---


,count,whitespace_percent
product_id,0.0,0.0
product_category_name,0.0,0.0



--- Missing Values Check ---


,count,null_percent
product_id,0,0.000000
product_category_name,610,1.851234
product_name_lenght,610,1.851234
product_description_lenght,610,1.851234
product_photos_qty,610,1.851234
product_weight_g,2,0.006070
product_length_cm,2,0.006070
product_height_cm,2,0.006070
product_width_cm,2,0.006070


# 📊 Data Quality Report: `df['olist_products_dataset']`

### 💡 Key Insights & "The Why" behind the Missing Values:

**1. The "Clustered" Nulls (The 610 rows):**
Look closely at the Missing Values report. `product_category_name`, `product_name_length`, `product_description_length`, and `product_photos_qty` all have **exactly 610 nulls**. 
This is not random! It means 610 specific products simply haven't been fully cataloged by the seller yet. 
*   **Action:** **DO NOT DROP THESE ROWS!** They are valid products that just lack descriptive metadata. If you drop them, you lose 610 products from your catalog that might still have orders. Leave them as `NaN`.

**2. The 2 Missing Measurements:**
Only **2 rows** are missing `product_weight_g`, `product_length_cm`, `product_height_cm`, and `product_width_cm`. This is likely a data entry error. 
*   **Action:** You can safely leave them as `NaN` (or fill them with the median weight if you later need them for shipping cost calculations).

**3. The "Float Trap" (Why are counts stored as Float?):**
`product_name_length`, `product_photos_qty`, and `product_weight_g` are showing as `float64` instead of `int64`. This is because Pandas *automatically* converts integers to floats when there are `NaN` (missing) values in the column to allow for the missing numbers. 
*   **Action:** Once you are done exploring, convert these measurement columns to integers using pandas' nullable integer type: `df['products'][col] = df['products'][col].astype('Int64')`. (Note the capital 'I' to allow NaN!).

---

### 🚀 Next Step:
You do not need to clean the nulls yet. The main action here is:
1. Leave the 610 nulls alone.
2. Convert the physical measurements (`weight`, `length`, `height`, `width`) to `'Int64'`.
3. Merge this table with the `order_items` table so you can analyze which product categories are bought the most.

In [26]:
data_quality_report(df[ 'olist_sellers_dataset'])

--- Data Quality Report ---


---shape----
(3095, 4)


---info---
<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 230.3 KB


--- Duplicate Check ---


,count,dup_percent
0,0,0.0





--- Whitespace Check ---


,count,whitespace_percent
seller_id,0.0,0.0
seller_city,0.0,0.0
seller_state,0.0,0.0



--- Missing Values Check ---


,count,null_percent
seller_id,0,0.0
seller_zip_code_prefix,0,0.0
seller_city,0,0.0
seller_state,0,0.0


# 📊 Data Quality Report: `df['olist_sellers_dataset']`

### 💡 Key Insights & "The Why" behind this table:

**1. The "Zero" Cleanup:**
This table requires absolutely **zero cleaning**. It is a perfectly structured dimension/reference table that has been maintained very well.

**2. The "Zip Code Trap" (Crucial Warning):**
Notice that `seller_zip_code_prefix` is `int64`. Just like the `geolocation` table, this is a trap!
If a seller's zip code starts with a `0` (e.g., `01234`), Pandas will store it as `1234` and drop the leading zero.
*   **Action:** **Do not leave it as `int64`!** Convert it to a string (`str`) or pad it with zeros before merging it with the orders or customers table, or you will fail to match them.
    ```python
    df['sellers']['seller_zip_code_prefix'] = df['sellers']['seller_zip_code_prefix'].astype(str)

In [27]:
data_quality_report(df[ 'product_category_name_translation'])

--- Data Quality Report ---


---shape----
(71, 2)


---info---
<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          71 non-null     str  
 1   product_category_name_english  71 non-null     str  
dtypes: str(2)
memory usage: 3.5 KB


--- Duplicate Check ---


,count,dup_percent
0,0,0.0





--- Whitespace Check ---


,count,whitespace_percent
product_category_name,0.0,0.0
product_category_name_english,0.0,0.0



--- Missing Values Check ---


,count,null_percent
product_category_name,0,0.0
product_category_name_english,0,0.0


# 📊 Data Quality Report: `df['product_category_name_translation']`

### 💡 Key Insights & "The Why" behind this table:

**1. The Role of this Table:**
This is a **Dictionary/Lookup Table** (Dimension Table). It contains only 71 rows because there are 71 unique product categories in the Olist dataset.
*   **The "Old" Language (Portuguese):** `product_category_name` (e.g., `"beleza_saude"`).
*   **The "New" Language (English):** `product_category_name_english` (e.g., `"health_beauty"`).

**2. Why is it useless for Analysis?**
If you try to run a direct `sum_whitespace` or `sum_nulls` on it, it looks perfect because it is a small reference table. But its main value is being **joined** to the `products` table later!
*   **Action:** Merge this table with `df['products']` using `product_category_name` to translate the Portuguese category names into English so you can present your findings to a global audience.

**3. The Duplicate Warning:**
Just like `order_items` and `payments`, **DO NOT DROP DUPLICATES ON `product_category_name` HERE** (even though there are none). It is a Primary Key. There are exactly 71 unique categories.

---

### 🚀 Next Step:
You have now successfully inspected **ALL 8 tables** in the Olist dataset using your custom `data_quality_report` function. 

**Now, you can begin the "Cleaning and Merging" phase:**
1. Translate the categories in `products`.
2. Convert all dates to `datetime` in `orders` and `reviews`.
3. Handle the zip codes (convert to `str`).
4. **Merge everything together:** `customers` + `orders` + `order_items` + `payments` + `products` + `reviews` + `sellers` + `geolocation` to create your final master analysis table!

# The Cleaning Face

# Utility Functions

# Function: `convert_to_datetime`

### Why do we need this function?
- In our manual inspection, we found that the timestamp columns in `orders`, `order_items`, and `reviews` are currently stored as `str` (strings).
- To calculate delivery times, order delays, or time gaps, we MUST convert them to `datetime` objects.

### Why `pd.to_datetime()`?
- It automatically parses strings (e.g., "2018-01-01 12:00:00") into Python's native datetime format, allowing us to perform arithmetic (subtracting dates) and time-series analysis.

### Why a `for` loop?
- Instead of writing `df['col1'] = pd.to_datetime(df['col1'])` multiple times, a loop allows us to pass a list of columns to convert them all at once. This is DRY (Don't Repeat Yourself).

### Why `return df`?
- Returning the modified DataFrame allows us to assign the result back to the dictionary (e.g., `df['orders'] = convert_to_datetime(df['orders'], cols)`), ensuring the changes are saved.

In [28]:

def convert_to_datetime(df, cols):
    for col in cols:
        df[col] = pd.to_datetime(df[col])
    return df




# Function: `clean_whitespace`

### Why do we need this function?
- During our Quality Report for the `geolocation` table, we found that some entries in `geolocation_city` had leading or trailing spaces (e.g., `" Sao Paulo "`).
- Extra spaces cause mismatches when merging tables (e.g., `"Sao Paulo"` will not match `" Sao Paulo"`).

### Why `.str.strip()`?
- The `.str` accessor tells Pandas to apply string operations to the entire column.
- `.strip()` removes whitespace from the beginning and end of the string (e.g., `" city "` becomes `"city"`). Note: It does **not** remove spaces *inside* the text (e.g., "New York" stays "New York").

### Why a `for` loop?
- Allows us to easily clean multiple string columns in one single call (e.g., cleaning `geolocation_city` and `geolocation_state` at the same time).

### Why `return df`?
- To keep the modified DataFrame updated in our main dictionary.

In [29]:
def clean_whitespace(df, cols):
    for col in cols:
        df[col] = df[col].str.strip()
    return df

# Function: `convert_zip_to_str`

### Why do we need this function?
- The `zip_code_prefix` columns in `customers`, `sellers`, and `geolocation` are currently stored as `int64`.
- **Crucial Note:** If a zip code starts with a `0` (e.g., `01234`), Pandas will drop the leading zero and store it as `1234`.
- If we merge tables using zip codes and one side has `1234` while the other has `01234`, the merge will fail!

### Why `.astype(str)`?
- Converting the column to strings preserves the leading zeros, ensuring that zip codes match correctly across different tables.

### Why a single `col` argument?
- Unlike the previous two functions, we usually convert zip codes one by one, because we need to specify exactly which table we are fixing (e.g., `df['customers']` vs `df['sellers']`).

### Why `return df`?
- To update the table in the main dictionary with the corrected string data type.

In [30]:
def convert_zip_to_str(df, col):
    df[col] = df[col].astype(str)
    return df

# Table-Specific Function

# Table-Specific Function: `aggregate_geolocation`

### Why is this a "specific" function?
- Unlike `convert_to_datetime`, this function **cannot** be applied to any other table because it is exclusively designed for the `geolocation` table.

### Why `groupby` instead of `drop_duplicates`?
- The geolocation table has 1 million rows and 26% duplicates, but these are **valid duplicates** (different GPS points inside the same zip code). We cannot delete them.
- The solution is to aggregate: we take the **mean** for latitude and longitude per zip code, and the **first** city/state. This reduces the table from 1 million rows to roughly 19,000 rows (1 row per zip code).

### Why `.reset_index()`?
- After `groupby`, the zip code becomes the index. We use `reset_index()` to turn it back into a regular column so it is ready to be merged with the `customers` table later.

### Why `return df`?
- Because the result is a completely new table, it must be returned and saved back into the main dictionary to replace the old (large) table.

In [31]:
def aggregate_geolocation(df):
    df = df.groupby('geolocation_zip_code_prefix').agg({
        'geolocation_lat': 'mean',
        'geolocation_lng': 'mean',
        'geolocation_city': 'first',
        'geolocation_state': 'first'
    }).reset_index()
    return df

# Table-Specific Function: `clean_products`

### Why is this a "specific" function?
- This function is exclusively for the `products` table, as it addresses its unique problems.

### Why `fillna('Unknown')`?
- The Quality Report showed there were 610 missing categories. These are real products, so we do not drop them. Instead, we label them as "Unknown" to keep the data intact.

### Why `.astype('Int64')` with a capital 'I'?
- Columns like length and weight currently show as `float64` (showing `0.0`) because Pandas automatically converts them when there are `NaN` values.
- Using capital `'Int64'` (Pandas' nullable integer type) allows us to convert them to whole numbers (`0`) **while keeping the missing values** as `NaN`. (If you used lowercase `int`, the code would crash because it cannot hold missing values).

### Why the `for` loop?
- To convert all 7 numeric columns at once, keeping the code clean and concise instead of writing 7 separate lines.

### Why write as a function instead of just raw commands?
- Because you are building a professional data pipeline. Putting these commands into a function allows you to easily re-run the cleaning later, and makes it seamless to call inside your master `run_all_cleaning` function without cluttering the notebook.

In [32]:
def clean_products(df):
   
    df = df.copy()

    df.columns = df.columns.str.strip()
  
    df['product_category_name'] = df['product_category_name'].fillna('Unknown')
    
    numeric_cols = [
        'product_name_lenght',  
        'product_description_lenght', 
        'product_photos_qty',
        'product_weight_g', 
        'product_length_cm', 
        'product_height_cm', 
        'product_width_cm'
    ]
    
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
            
    return df

In [33]:
def translate_categories(df_products, df_translation):
    
    df_merged = df_products.merge(
        df_translation, 
        on='product_category_name', 
        how='left'
    )
    
   
    df_merged['product_category_name'] = df_merged['product_category_name_english'].fillna(df_merged['product_category_name'])
    
    
    df_merged = df_merged.drop(columns=['product_category_name_english'])
    
    return df_merged

# Cleaning Step 1: Orders Table

### Why clean this table first?
- `orders` is the central table (Master Fact Table) for this dataset. All other tables will eventually be merged to it.
- Our Quality Report showed that all timestamp columns are `str`. We must convert them to `datetime` to calculate delivery times and delays.

### How to verify?
- After running the code, we will check `df['orders'].dtypes` to confirm the data types have changed from `object` to `datetime64[ns]`.

In [34]:
# Define the columns containing dates
orders_date_cols = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

# Apply the generic function to clean dates
df['olist_orders_dataset'] = convert_to_datetime(df['olist_orders_dataset'], orders_date_cols)

# Verify the changes
print(df['olist_orders_dataset'].dtypes)

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [35]:
df.keys()

dict_keys(['customers', 'olist_geolocation_dataset', 'olist_orders_dataset', 'olist_order_items_dataset', 'olist_order_payments_dataset', 'olist_order_reviews_dataset', 'olist_products_dataset', 'olist_sellers_dataset', 'product_category_name_translation'])

# Cleaning Step 2: Order Items Table

### Why clean this table?
- This table has 1 date column (`shipping_limit_date`) stored as a string.
- Converting this allows us to compare the shipping deadline against when the item was actually shipped.

In [36]:
# Convert the shipping limit date
df['olist_order_items_dataset'] = convert_to_datetime(df['olist_order_items_dataset'], ['shipping_limit_date'])

# Verify
print(df['olist_order_items_dataset'].dtypes)

order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object


# Cleaning Step 3: Reviews Table

### Why clean this table?
- The review dates (`review_creation_date` and `review_answer_timestamp`) are strings.
- Converting them allows us to analyze how long it takes sellers to respond to reviews.

In [37]:
# Convert review dates
df['olist_order_reviews_dataset'] = convert_to_datetime(df['olist_order_reviews_dataset'], ['review_creation_date', 'review_answer_timestamp'])

# Verify
print(df['olist_order_reviews_dataset'].dtypes)

review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object


# Cleaning Step 4: Customers & Sellers (Zip Codes)

### Why convert to String?
- Zip codes are stored as `int64`.
- If a zip code starts with a 0 (e.g., 01234), Pandas drops the zero (stores it as 1234).
- Converting to string preserves the leading zero, ensuring they match perfectly when we merge the tables.

In [38]:
# Convert zip codes to strings
df['customers'] = convert_zip_to_str(df['customers'], 'customer_zip_code_prefix')
df['olist_sellers_dataset'] = convert_zip_to_str(df['olist_sellers_dataset'], 'seller_zip_code_prefix')

# Verify
print("Customers zip dtype:", df['customers']['customer_zip_code_prefix'].dtype)
print("Sellers zip dtype:", df['olist_sellers_dataset']['seller_zip_code_prefix'].dtype)

Customers zip dtype: str
Sellers zip dtype: str


# Cleaning Step 5: Geolocation Table

### The Specific Cleaning Steps:
1. **Aggregate:** Reduce 1,000,163 rows to ~19,000 by grouping by zip code and taking the mean lat/lng.
2. **Zip Code:** Convert the new zip column to string.
3. **Whitespace:** Clean the `geolocation_city` column for extra spaces.

In [39]:
# 1. Aggregate by Zip Code
df['olist_geolocation_dataset'] = aggregate_geolocation(df['olist_geolocation_dataset'])

# 2. Convert Zip to String (Must be done AFTER aggregation)
df['olist_geolocation_dataset'] = convert_zip_to_str(df['olist_geolocation_dataset'], 'geolocation_zip_code_prefix')

# 3. Clean Whitespace in City
df['olist_geolocation_dataset'] = clean_whitespace(df['olist_geolocation_dataset'], ['geolocation_city'])

# Verify shape and dtypes
print("New Geolocation Shape:", df['olist_geolocation_dataset'].shape)
print(df['olist_geolocation_dataset'].dtypes)

New Geolocation Shape: (19015, 5)
geolocation_zip_code_prefix        str
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object


# Cleaning Step 6: Products Table

### The Specific Cleaning Steps:
1. **Fill Missing Categories:** Fill the 610 missing `product_category_name` with "Unknown".
2. **Convert Numeric Columns:** Change `float64` to `Int64` (whole numbers) without losing the missing values (NaN).
3. **Translate:** Merge the products table with the English translation table.

In [40]:
# 1. Clean the specific product issues
df['olist_products_dataset'] = clean_products(df['olist_products_dataset'])

# 2. Translate categories to English
df['olist_products_dataset'] = translate_categories(df['olist_products_dataset'], df['product_category_name_translation'])

# Verify
print(df['olist_products_dataset'].dtypes)

product_id                      str
product_category_name           str
product_name_lenght           Int64
product_description_lenght    Int64
product_photos_qty            Int64
product_weight_g              Int64
product_length_cm             Int64
product_height_cm             Int64
product_width_cm              Int64
dtype: object


In [41]:
print(df['olist_products_dataset'].columns)

Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='str')


# Save Cleaned Tables for Advanced EDA

### Why use Parquet instead of CSV?
- Parquet preserves the exact data types (e.g., `datetime`, `Int64`) without needing to re-convert them.
- It is significantly faster to load and uses much less disk space than CSV.
- We save each table separately so we can load only what we need in the new notebook.

In [42]:
eda_folder = r'D:\olist-ecommerce-analytics\notebooks\data\eda'

for table_name, table_df in df.items():
    table_df.to_parquet(f'{eda_folder}/{table_name}.parquet', index=False, engine='fastparquet')
    print(f"Successfully saved: {table_name}")

Successfully saved: customers
Successfully saved: olist_geolocation_dataset
Successfully saved: olist_orders_dataset
Successfully saved: olist_order_items_dataset
Successfully saved: olist_order_payments_dataset
Successfully saved: olist_order_reviews_dataset
Successfully saved: olist_products_dataset
Successfully saved: olist_sellers_dataset
Successfully saved: product_category_name_translation


In [43]:
df['olist_orders_dataset'].info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 13.0 MB


In [44]:
df.keys()

dict_keys(['customers', 'olist_geolocation_dataset', 'olist_orders_dataset', 'olist_order_items_dataset', 'olist_order_payments_dataset', 'olist_order_reviews_dataset', 'olist_products_dataset', 'olist_sellers_dataset', 'product_category_name_translation'])

In [45]:
%pip install --upgrade pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [48]:
import os
import pandas as pd
from sqlalchemy import (
    BigInteger,
    Boolean,
    DateTime,
    Float,
    NVARCHAR,
    URL,
    create_engine,
)

# 1. Database Configuration & Engine Setup
server = r"localhost\SQL2025"
database = "olist_data"
driver = "ODBC Driver 17 for SQL Server"

connection_string = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
)

engine = create_engine(
    URL.create(
        "mssql+pyodbc",
        query={"odbc_connect": connection_string},
    ),
    fast_executemany=True,
)

# 2. Create the Bronze Schema
with engine.begin() as connection:
    connection.exec_driver_sql("""
        IF SCHEMA_ID(N'bronze') IS NULL
        BEGIN
            EXEC(N'CREATE SCHEMA [bronze]')
        END
    """)

# 3. Data Mapping Function
def get_sql_dtype_mapping(dataframe: pd.DataFrame) -> dict:
    """Map Pandas data types to SQL Server data types."""
    dtype_mapping = {}

    for column in dataframe.columns:
        series = dataframe[column]

        if pd.api.types.is_datetime64_any_dtype(series):
            dtype_mapping[column] = DateTime()
        elif pd.api.types.is_bool_dtype(series):
            dtype_mapping[column] = Boolean()
        elif pd.api.types.is_integer_dtype(series):
            dtype_mapping[column] = BigInteger()
        elif pd.api.types.is_float_dtype(series):
            dtype_mapping[column] = Float()
        else:
            dtype_mapping[column] = NVARCHAR(length=None)

    return dtype_mapping

# 4. Table Upload Function (With FK Drop Handling)
def upload_dataframe_to_sql(dataframe: pd.DataFrame, table_name: str, schema: str = "bronze"):
    """Safely drop constraints, replace a SQL Server table, and upload a DataFrame."""
    if not isinstance(dataframe, pd.DataFrame):
        raise TypeError(f"{table_name} is not a Pandas DataFrame")

    dataframe = dataframe.copy()
    dataframe.columns = dataframe.columns.astype(str)

    if dataframe.columns.duplicated().any():
        duplicated_columns = dataframe.columns[dataframe.columns.duplicated()].tolist()
        raise ValueError(
            f"Duplicate column names found in {table_name}: {duplicated_columns}"
        )

    # Clean up FK references before attempting to drop the target table
    with engine.begin() as conn:
        conn.exec_driver_sql(f"""
            IF OBJECT_ID(N'[{schema}].[{table_name}]', 'U') IS NOT NULL
            BEGIN
                DECLARE @sql NVARCHAR(MAX) = '';
                SELECT @sql += 'ALTER TABLE ' + QUOTENAME(OBJECT_SCHEMA_NAME(parent_object_id)) 
                    + '.' + QUOTENAME(OBJECT_NAME(parent_object_id)) 
                    + ' DROP CONSTRAINT ' + QUOTENAME(name) + ';'
                FROM sys.foreign_keys
                WHERE referenced_object_id = OBJECT_ID(N'[{schema}].[{table_name}]');
                
                EXEC sp_executesql @sql;
                DROP TABLE [{schema}].[{table_name}];
            END
        """)

    # Upload DataFrame with schema creation
    dataframe.to_sql(
        name=str(table_name),
        con=engine,
        schema=schema,
        if_exists="fail",
        index=False,
        dtype=get_sql_dtype_mapping(dataframe),
        chunksize=1000,
        method=None,
    )

    print(f"Uploaded [{schema}].[{table_name}] ({len(dataframe.columns)} cols, {len(dataframe):,} rows)")

# 5. Execution Loop
for table_name, dataframe in df.items():
    upload_dataframe_to_sql(
        dataframe=dataframe,
        table_name=table_name,
        schema="bronze",
    )

print("All DataFrames were uploaded successfully.")

Uploaded [bronze].[customers] (5 cols, 99,441 rows)
Uploaded [bronze].[olist_geolocation_dataset] (5 cols, 19,015 rows)
Uploaded [bronze].[olist_orders_dataset] (8 cols, 99,441 rows)
Uploaded [bronze].[olist_order_items_dataset] (7 cols, 112,650 rows)
Uploaded [bronze].[olist_order_payments_dataset] (5 cols, 103,886 rows)
Uploaded [bronze].[olist_order_reviews_dataset] (7 cols, 99,224 rows)
Uploaded [bronze].[olist_products_dataset] (9 cols, 32,951 rows)
Uploaded [bronze].[olist_sellers_dataset] (4 cols, 3,095 rows)
Uploaded [bronze].[product_category_name_translation] (2 cols, 71 rows)
All DataFrames were uploaded successfully.
